In [1]:
import sqlite3

# 1. Database Setup
conn = sqlite3.connect("library.db")
cursor = conn.cursor()

cursor.execute(
    """
CREATE TABLE IF NOT EXISTS books (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    title TEXT NOT NULL,
    author TEXT NOT NULL,
    status TEXT DEFAULT 'Available'
)
"""
)
conn.commit()


# 2. Core Functions
def add_book(title, author):
    cursor.execute(
        "INSERT INTO books (title, author) VALUES (?, ?)", (title, author)
    )
    conn.commit()
    print(f"Added: '{title}' by {author}")


def view_books():
    cursor.execute("SELECT * FROM books")
    for row in cursor.fetchall():
        print(f"ID: {row[0]} | Title: {row[1]} | Author: {row[2]} | Status: {row[3]}")


def issue_book(book_id):
    cursor.execute(
        "UPDATE books SET status = 'Issued' WHERE id = ? AND status = 'Available'",
        (book_id,),
    )
    if cursor.rowcount > 0:
        conn.commit()
        print(f"Book ID {book_id} issued successfully.")
    else:
        print(f"Book ID {book_id} is unavailable or does not exist.")


def return_book(book_id):
    cursor.execute(
        "UPDATE books SET status = 'Available' WHERE id = ? AND status = 'Issued'",
        (book_id,),
    )
    if cursor.rowcount > 0:
        conn.commit()
        print(f"Book ID {book_id} returned successfully.")
    else:
        print(f"Book ID {book_id} was not issued or does not exist.")


# 3. Execution Example
add_book("Clean Code", "Robert C. Martin")
add_book("The Pragmatic Programmer", "Andrew Hunt")

print("\n--- All Books ---")
view_books()

print("\n--- Issuing Book ---")
issue_book(1)

print("\n--- Updated Inventory ---")
view_books()

# conn.close()

Added: 'Clean Code' by Robert C. Martin
Added: 'The Pragmatic Programmer' by Andrew Hunt

--- All Books ---
ID: 1 | Title: Clean Code | Author: Robert C. Martin | Status: Available
ID: 2 | Title: The Pragmatic Programmer | Author: Andrew Hunt | Status: Available

--- Issuing Book ---
Book ID 1 issued successfully.

--- Updated Inventory ---
ID: 1 | Title: Clean Code | Author: Robert C. Martin | Status: Issued
ID: 2 | Title: The Pragmatic Programmer | Author: Andrew Hunt | Status: Available


In [2]:
def search_books(cursor, keyword):
    cursor.execute(
        "SELECT * FROM books WHERE title LIKE ? OR author LIKE ?",
        (f"%{keyword}%", f"%{keyword}%"),
    )
    results = cursor.fetchall()
    return results if results else "No books found matching criteria."

In [3]:
def check_availability(cursor, book_id):
    cursor.execute("SELECT title, status FROM books WHERE id = ?", (book_id,))
    record = cursor.fetchone()
    if record:
        return f"'{record[0]}' is currently: {record[1]}"
    return "Book ID not found."

In [4]:
from datetime import datetime


def calculate_fine(due_date_str, daily_rate=5):
    due_date = datetime.strptime(due_date_str, "%Y-%m-%d")
    today = datetime.now()
    overdue_days = (today - due_date).days

    if overdue_days > 0:
        return overdue_days * daily_rate
    return 0  # No fine if returned on or before due date

In [5]:
print("--- Function 1: Search Books ---")
search_results = search_books(cursor, "Python")
print(search_results)

# Call 2: Check availability for Book ID 2
print("\n--- Function 2: Check Availability ---")
status_message = check_availability(cursor, 2)
print(status_message)

# Call 3: Calculate fine for an overdue date (e.g., due on August 20, 2026)
print("\n--- Function 3: Calculate Fine ---")
fine_amount = calculate_fine("2026-08-20", daily_rate=5)
print(f"Total Fine: ₹{fine_amount}")

# Close connection
conn.close()

--- Function 1: Search Books ---
No books found matching criteria.

--- Function 2: Check Availability ---
'The Pragmatic Programmer' is currently: Available

--- Function 3: Calculate Fine ---
Total Fine: ₹80
